# Mask R-CNN + EfficientNet-FPN — Segmentación de Vértebras

Segmentación multiclase de vértebras en radiografías de columna.  
**Clases:** 23 (background + C3–C7 + T1–T12 + L1–L5)  
**Prioridad:** Baja — exploración opcional tras ResNet50 y nnU-Net.  
**Nota:** EfficientNet no es backbone nativo de Mask R-CNN en torchvision. Se integra via `timm` + FPN personalizado.  
**Dependencia extra:** `pip install timm`

In [17]:
import os
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models.detection import MaskRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import FeaturePyramidNetwork
from torchvision.ops.feature_pyramid_network import LastLevelMaxPool
import timm
import albumentations as A
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import albumentations as A
from sklearn.model_selection import train_test_split
from pathlib import Path


In [18]:
# === CONFIGURACIÓN ===

#DATASET_ROOT    = '../MaIA_Scoliosis_Dataset'
DATASET_ROOT = Path(r"C:\Users\USER\Scoliosis_Dataset")
DATASET_INDEX = DATASET_ROOT / "indice_dataset.csv"
#DATASET_INDEX   = os.path.join(DATASET_ROOT, 'dataset_index.csv')
CHECKPOINTS_DIR = 'checkpoints_efficientnet'
MODELS_DIR      = '../models'

DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TARGET_SIZE  = (512, 1024)
NUM_CLASSES  = 23
BASE_LR      = 1e-3
WEIGHT_DECAY = 1e-4
BATCH_SIZE   = 1
SEED         = 42

CLASS_NAMES = {
    1: 'C7',  2: 'C6',  3: 'C5',  4: 'C4',  5: 'C3',
    6: 'T1',  7: 'T2',  8: 'T3',  9: 'T4',  10: 'T5',
    11: 'T6', 12: 'T7', 13: 'T8', 14: 'T9', 15: 'T10',
    16: 'T11', 17: 'T12',
    18: 'L1', 19: 'L2', 20: 'L3', 21: 'L4', 22: 'L5',
}

os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f'Dispositivo: {DEVICE}')

Dispositivo: cpu


In [19]:
print(os.listdir(DATASET_ROOT))

['diccionario_etiquetas_T1_T12_L1_L5.json', 'indice_dataset.csv', 'LabelBinaryJPG', 'LabelMultiClass_Color_JPG', 'LabelMultiClass_Gray_JPG', 'LabelMultiClass_ID_PNG', 'Normal', 'RadiographMetrics', 'README.md', 'reporte_por_mascara_version_final.csv', 'resumen_diccionario_original_35_entidades.csv', 'resumen_version_final_T1_T12_L1_L5.csv', 'Scoliosis']


In [20]:
df = pd.read_csv(DATASET_INDEX)
print(df.head())
print(df.columns)

  split;image;patient_id;radiograph_path;label_binary_path;multiclass_id_png;multiclass_gray_jpg;multiclass_color_jpg;metrics_json
0  Normal;N_1.jpg;1;Normal/N_1.jpg;LabelBinaryJPG...                                                                              
1  Normal;N_2.jpg;2;Normal/N_2.jpg;LabelBinaryJPG...                                                                              
2  Normal;N_3.jpg;3;Normal/N_3.jpg;LabelBinaryJPG...                                                                              
3  Normal;N_4.jpg;4;Normal/N_4.jpg;LabelBinaryJPG...                                                                              
4  Normal;N_5.jpg;5;Normal/N_5.jpg;LabelBinaryJPG...                                                                              
Index(['split;image;patient_id;radiograph_path;label_binary_path;multiclass_id_png;multiclass_gray_jpg;multiclass_color_jpg;metrics_json'], dtype='object')


---
## Sección 1 — Preprocesamiento

### Funciones de carga

In [21]:
def load_image(image_path: str) -> np.ndarray:
    """Carga imagen RGB uint8 desde disco."""
    return np.array(Image.open(image_path).convert('RGB'))


def load_mask(mask_path: str) -> np.ndarray:
    """Carga máscara 16-bit uint16 sin truncar IDs de clase."""
    return cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)


def load_binary_mask(binary_mask_path: str) -> np.ndarray:
    """Carga máscara binaria y la binariza para neutralizar artefactos JPEG."""
    raw = cv2.imread(binary_mask_path, cv2.IMREAD_GRAYSCALE)
    return (raw > 127).astype(np.uint8)


def load_dataset_index(csv_path: str) -> pd.DataFrame:
    """Carga dataset_index.csv como fuente canónica de rutas."""
    return pd.read_csv(csv_path)

### Transformaciones

In [22]:
def to_grayscale(image: np.ndarray) -> np.ndarray:
    """Conversión perceptual: L = 0.299R + 0.587G + 0.114B."""
    return (0.299 * image[:, :, 0]
            + 0.587 * image[:, :, 1]
            + 0.114 * image[:, :, 2]).astype(np.uint8)


def map_entity_ids(mask: np.ndarray) -> np.ndarray:
    """Mapea IDs 23–35 (Entity X) a 0 (background). Devuelve uint8."""
    result = mask.copy()
    result[result > 22] = 0
    return result.astype(np.uint8)


def compute_roi(binary_mask: np.ndarray, margin: float = 0.10) -> tuple:
    """Bounding box de la columna con margen ≥10%."""
    rows = np.any(binary_mask, axis=1)
    cols = np.any(binary_mask, axis=0)
    if not rows.any():
        h, w = binary_mask.shape
        return (0, 0, w, h)
    y1, y2 = np.where(rows)[0][[0, -1]]
    x1, x2 = np.where(cols)[0][[0, -1]]
    h, w = binary_mask.shape
    dy = max(1, int((y2 - y1) * margin))
    dx = max(1, int((x2 - x1) * margin))
    return (max(0, x1 - dx), max(0, y1 - dy), min(w, x2 + dx), min(h, y2 + dy))


def crop_to_roi(image: np.ndarray, mask: np.ndarray, roi: tuple) -> tuple:
    """Aplica el mismo crop a imagen y máscara."""
    x1, y1, x2, y2 = roi
    return image[y1:y2, x1:x2], mask[y1:y2, x1:x2]


def resize_pair(image: np.ndarray, mask: np.ndarray, target_size: tuple) -> tuple:
    """Resize: bilinear para imagen, nearest neighbor para máscara."""
    w, h = target_size
    return (cv2.resize(image, (w, h), interpolation=cv2.INTER_LINEAR),
            cv2.resize(mask,  (w, h), interpolation=cv2.INTER_NEAREST))


def apply_clahe(image: np.ndarray, clip_limit=2.0, tile_grid=(8, 8)) -> np.ndarray:
    """CLAHE sobre imagen monocanal uint8."""
    return cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid).apply(image)


def replicate_to_3ch(image: np.ndarray) -> np.ndarray:
    """Replica canal gris (H,W) a 3 canales (H,W,3)."""
    return np.stack([image, image, image], axis=-1)


def normalize_image(image: np.ndarray) -> torch.Tensor:
    """Estandarización con stats ImageNet. Entrada uint8 (H,W,3) → Tensor float32 (3,H,W)."""
    img  = image.astype(np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    return torch.from_numpy(((img - mean) / std).transpose(2, 0, 1))


def semantic_to_instance(mask: np.ndarray) -> dict:
    """Convierte máscara semántica a formato instancia para Mask R-CNN."""
    class_ids = np.unique(mask)
    class_ids = class_ids[class_ids > 0]
    boxes, masks, labels = [], [], []
    for cid in class_ids:
        binary = (mask == cid).astype(np.uint8)
        ys, xs = np.where(binary)
        if len(xs) == 0:
            continue
        x1, y1, x2, y2 = float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())
        if x2 <= x1 or y2 <= y1:
            continue
        boxes.append([x1, y1, x2, y2])
        masks.append(binary)
        labels.append(int(cid))
    if not boxes:
        h, w = mask.shape
        return {
            'boxes':  torch.zeros((0, 4), dtype=torch.float32),
            'masks':  torch.zeros((0, h, w), dtype=torch.uint8),
            'labels': torch.zeros(0, dtype=torch.int64),
        }
    return {
        'boxes':  torch.tensor(boxes, dtype=torch.float32),
        'masks':  torch.tensor(np.stack(masks), dtype=torch.uint8),
        'labels': torch.tensor(labels, dtype=torch.int64),
    }

### Augmentation

In [23]:
def build_augmentation_pipeline() -> A.Compose:
    """Pipeline de augmentation sincronizada imagen-máscara."""
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=10, border_mode=cv2.BORDER_CONSTANT, value=0, mask_value=0, p=0.5),
        A.RandomScale(scale_limit=0.2, p=0.4),
        A.ElasticTransform(alpha=1, sigma=50, p=0.3),
        A.RandomBrightnessContrast(p=0.4),
    ], additional_targets={'mask': 'mask'})


def apply_augmentation(image, mask, pipeline) -> tuple:
    """Augmentation sincronizada imagen-máscara."""
    result = pipeline(image=image, mask=mask)
    return result['image'], result['mask']

### Split y Dataset

In [24]:


class SpineDataset(Dataset):
    """Dataset de vértebras con pipeline de preprocesamiento completo."""

    def __init__(self, df, mode, aug_pipeline=None, target_size=TARGET_SIZE, dataset_root=DATASET_ROOT):
        self.df, self.mode, self.aug_pipeline = df, mode, aug_pipeline
        self.target_size, self.root = target_size, dataset_root

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row         = self.df.iloc[idx]
        image       = load_image(os.path.join(self.root, row['radiograph_path']))
        mask        = load_mask(os.path.join(self.root, row['multiclass_id_png']))
        binary_mask = load_binary_mask(os.path.join(self.root, row['label_binary_path']))

        image = to_grayscale(image)
        mask  = map_entity_ids(mask)
        roi   = compute_roi(binary_mask)
        image, mask = crop_to_roi(image, mask, roi)
        image, mask = resize_pair(image, mask, self.target_size)
        image = apply_clahe(image)
        if self.mode == 'train' and self.aug_pipeline is not None:
            image, mask = apply_augmentation(image, mask, self.aug_pipeline)

        image_tensor        = normalize_image(replicate_to_3ch(image))
        target              = semantic_to_instance(mask)
        target['full_mask'] = torch.from_numpy(mask.copy())
        target['image_id']  = torch.tensor([idx])
        return image_tensor, target


def collate_fn(batch):
    """Mask R-CNN espera lista de tensores, no batch tensorial apilado."""
    return [item[0] for item in batch], [item[1] for item in batch]


def create_dataloaders(train_ds, val_ds, test_ds, batch_size=BATCH_SIZE):
    """Crea los 3 DataLoaders con collate_fn para Mask R-CNN."""
    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True,  collate_fn=collate_fn, num_workers=2),
        DataLoader(val_ds,   batch_size=batch_size, shuffle=False, collate_fn=collate_fn, num_workers=2),
        DataLoader(test_ds,  batch_size=batch_size, shuffle=False, collate_fn=collate_fn, num_workers=2),
    )

In [25]:
from sklearn.model_selection import train_test_split

def split_dataset(df, train=0.70, val=0.15, seed=42):
    train_df, temp_df = train_test_split(
        df,
        train_size=train,
        random_state=seed
    )

    val_ratio = val / (1.0 - train)

    val_df, test_df = train_test_split(
        temp_df,
        train_size=val_ratio,
        random_state=seed
    )

    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True)
    )

---
## Sección 2 — Procesamiento (Entrenamiento)

### Construcción del backbone EfficientNet-FPN

In [26]:
class EfficientNetFeatureExtractor(nn.Module):
    """
    Extractor de features multi-escala de EfficientNet-B4 via timm.
    Retorna dict {'0': f2, '1': f3, '2': f4} para compatibilidad con FPN.
    out_indices=(2,3,4) extrae stages 2, 3 y 4 de EfficientNet-B4.
    """

    def __init__(self, model_name: str = 'efficientnet_b4', out_indices: tuple = (2, 3, 4)):
        super().__init__()
        self.model = timm.create_model(
            model_name, pretrained=True, features_only=True, out_indices=out_indices
        )
        self.out_channels = self.model.feature_info.channels()

    def forward(self, x):
        features = self.model(x)
        return {str(i): f for i, f in enumerate(features)}


class EfficientNetWithFPN(nn.Module):
    """
    Backbone EfficientNet-B4 + FPN. Compatible con torchvision MaskRCNN.
    out_channels es requerido por MaskRCNN para dimensionar el RPN.
    """

    def __init__(self, extractor: EfficientNetFeatureExtractor,
                 fpn: FeaturePyramidNetwork, out_channels: int):
        super().__init__()
        self.body        = extractor
        self.fpn         = fpn
        self.out_channels = out_channels

    def forward(self, x):
        return self.fpn(self.body(x))


def build_efficientnet_backbone(model_name: str = 'efficientnet_b4',
                                out_channels: int = 256) -> EfficientNetWithFPN:
    """Construye EfficientNet-B4 con FPN. Pesos: ImageNet via timm."""
    extractor = EfficientNetFeatureExtractor(model_name)
    fpn = FeaturePyramidNetwork(
        in_channels_list=list(extractor.out_channels),
        out_channels=out_channels,
        extra_blocks=LastLevelMaxPool(),
    )
    return EfficientNetWithFPN(extractor, fpn, out_channels)


def build_model(num_classes: int = NUM_CLASSES) -> MaskRCNN:
    """
    Mask R-CNN con backbone EfficientNet-B4 + FPN.
    AnchorGenerator y ROI poolers configurados para 4 niveles de FPN
    (3 de EfficientNet stages + 1 de LastLevelMaxPool).
    """
    backbone = build_efficientnet_backbone()

    anchor_sizes   = ((32,), (64,), (128,), (256,))
    aspect_ratios  = ((0.5, 1.0, 2.0),) * len(anchor_sizes)
    anchor_gen     = AnchorGenerator(sizes=anchor_sizes, aspect_ratios=aspect_ratios)

    roi_pooler = torchvision.ops.MultiScaleRoIAlign(
        featmap_names=['0', '1', '2', '3'], output_size=7, sampling_ratio=2
    )
    mask_roi_pooler = torchvision.ops.MultiScaleRoIAlign(
        featmap_names=['0', '1', '2', '3'], output_size=14, sampling_ratio=2
    )

    model = MaskRCNN(
        backbone=backbone,
        num_classes=num_classes,
        rpn_anchor_generator=anchor_gen,
        box_roi_pool=roi_pooler,
        mask_roi_pool=mask_roi_pooler,
    )
    return model

### Control del encoder

In [27]:
def freeze_encoder(model) -> None:
    """Congela todos los parámetros del backbone EfficientNet-B4."""
    for param in model.backbone.body.model.parameters():
        param.requires_grad = False


def unfreeze_stage(model, stage_idx: int) -> None:
    """
    Descongela el stage indicado del backbone EfficientNet-B4.
    En timm, EfficientNet-B4 tiene 7 grupos de bloques (blocks[0]–blocks[6]).
    Llamar en orden descendente: 6 → 5 → 4.
    """
    for param in model.backbone.body.model.blocks[stage_idx].parameters():
        param.requires_grad = True


def get_param_groups(model, base_lr: float = BASE_LR) -> list:
    """
    Grupos de parámetros con LR diferencial por stage del encoder.
    FPN/RPN/cabezas: base_lr
    blocks[6]: base_lr × 0.1  |  blocks[5]: × 0.01  |  blocks[4]: × 0.001
    """
    stage_lr = {
        'backbone.body.model.blocks.6': base_lr * 0.1,
        'backbone.body.model.blocks.5': base_lr * 0.01,
        'backbone.body.model.blocks.4': base_lr * 0.001,
    }
    stage_params   = {k: [] for k in stage_lr}
    decoder_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        matched = False
        for stage_name in stage_lr:
            if name.startswith(stage_name):
                stage_params[stage_name].append(param)
                matched = True
                break
        if not matched:
            decoder_params.append(param)

    groups = []
    if decoder_params:
        groups.append({'params': decoder_params, 'lr': base_lr})
    for stage_name, params in stage_params.items():
        if params:
            groups.append({'params': params, 'lr': stage_lr[stage_name]})
    return groups

### Optimizador y scheduling

In [42]:
def build_optimizer(param_groups, weight_decay=WEIGHT_DECAY):
    """SGD con momentum=0.9 y weight_decay."""
    return torch.optim.SGD(param_groups, momentum=0.9, weight_decay=weight_decay)


def build_scheduler(optimizer, patience=3, factor=0.5):
    """Reduce LR a la mitad si val_loss no mejora en patience epochs."""
    return torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=factor,
        patience=patience
    )

### Loops de entrenamiento

In [29]:
def train_one_epoch(model, dataloader, optimizer, scaler, device) -> float:
    """Loop de entrenamiento con mixed precision (FP16)."""
    model.train()
    total_loss = 0.0
    for images, targets in dataloader:
        images = [img.to(device) for img in images]
        model_targets = [
            {k: v.to(device) for k, v in t.items() if k not in ('full_mask', 'image_id')}
            for t in targets
        ]
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            loss_dict = model(images, model_targets)
            losses    = sum(loss_dict.values())
        scaler.scale(losses).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += losses.item()
    return total_loss / len(dataloader)


def validate_one_epoch(model, dataloader, device) -> float:
    """Validación en modo train con no_grad para obtener losses de Mask R-CNN."""
    model.train()
    total_loss = 0.0
    with torch.no_grad():
        for images, targets in dataloader:
            images = [img.to(device) for img in images]
            model_targets = [
                {k: v.to(device) for k, v in t.items() if k not in ('full_mask', 'image_id')}
                for t in targets
            ]
            total_loss += sum(model(images, model_targets).values()).item()
    return total_loss / len(dataloader)

### Checkpointing

In [30]:
def save_checkpoint(model, path: str, val_loss: float) -> None:
    """Guarda pesos cuando val_loss mejora."""
    torch.save(model.state_dict(), path)
    print(f'  Checkpoint guardado: {path}  (val_loss={val_loss:.4f})')


def load_checkpoint(model, path: str):
    """Carga el mejor checkpoint de una fase."""
    model.load_state_dict(torch.load(path, map_location='cpu'))
    return model


def save_final_model(model, path: str) -> None:
    """Guarda el modelo final en un .pth fijo al terminar todas las fases."""
    torch.save(model.state_dict(), path)
    print(f'Modelo final guardado en: {path}')

### Entrenamiento por fases

In [36]:
def train_phase(model, train_dl, val_dl, optimizer, scheduler, device,
                max_epochs, patience=7, checkpoint_path='checkpoint.pth') -> float:
    """Fase completa: ReduceLROnPlateau + early stopping + checkpointing."""
    scaler, best_val_loss, epochs_no_impr = torch.cuda.amp.GradScaler(), float('inf'), 0
    for epoch in range(1, max_epochs + 1):
        train_loss = train_one_epoch(model, train_dl, optimizer, scaler, device)
        val_loss   = validate_one_epoch(model, val_dl, device)
        scheduler.step(val_loss)
        print(f'  Epoch {epoch}/{max_epochs} — train: {train_loss:.4f}  val: {val_loss:.4f}')
        if val_loss < best_val_loss:
            best_val_loss, epochs_no_impr = val_loss, 0
            save_checkpoint(model, checkpoint_path, val_loss)
        else:
            epochs_no_impr += 1
            if epochs_no_impr >= patience:
                print(f'  Early stopping en epoch {epoch}')
                break
    return best_val_loss


def run_progressive_training(model, train_dl, val_dl, device,
                             base_lr=BASE_LR, weight_decay=WEIGHT_DECAY):
    """
    3 fases de descongelamiento progresivo sobre stages de EfficientNet-B4:
      Fase 1 — Encoder congelado        (15 epochs max)
      Fase 2 — Descongelar blocks[6]    (10 epochs max)
      Fase 3 — Descongelar blocks[5]    ( 8 epochs max)
    """
    phases = [
        {'name': 'Fase 1 — Encoder congelado',   'unfreeze': None, 'max_epochs': 15},
        {'name': 'Fase 2 — Descongelar blocks[6]', 'unfreeze': 6,  'max_epochs': 10},
        {'name': 'Fase 3 — Descongelar blocks[5]', 'unfreeze': 5,  'max_epochs': 8},
    ]
    freeze_encoder(model)
    for i, phase in enumerate(phases, start=1):
        print(f'\n=== {phase["name"]} ===')
        if phase['unfreeze'] is not None:
            unfreeze_stage(model, phase['unfreeze'])
        param_groups = get_param_groups(model, base_lr)
        optimizer    = build_optimizer(param_groups, weight_decay)
        scheduler    = build_scheduler(optimizer)
        ckpt_path    = os.path.join(CHECKPOINTS_DIR, f'phase{i}_best.pth')
        best = train_phase(model, train_dl, val_dl, optimizer, scheduler,
                           device, phase['max_epochs'], checkpoint_path=ckpt_path)
        model = load_checkpoint(model, ckpt_path)
        print(f'  Mejor val_loss fase {i}: {best:.4f}')
    return model

---
## Sección 3 — Métricas

### Métricas de pixel (Dice e IoU)

In [37]:
def compute_dice(pred: np.ndarray, gt: np.ndarray) -> float:
    pred, gt = pred.astype(bool), gt.astype(bool)
    inter = (pred & gt).sum()
    denom = pred.sum() + gt.sum()
    return 2.0 * inter / denom if denom > 0 else 1.0


def compute_iou(pred: np.ndarray, gt: np.ndarray) -> float:
    pred, gt = pred.astype(bool), gt.astype(bool)
    inter = (pred & gt).sum()
    union = (pred | gt).sum()
    return inter / union if union > 0 else 1.0


def _scores_per_class(predictions, targets, metric_fn, num_classes):
    scores = {c: [] for c in range(1, num_classes)}
    for pred, target in zip(predictions, targets):
        full_mask   = target['full_mask'].numpy()
        pred_labels = pred['labels'].numpy()
        pred_masks  = pred['masks'].numpy()
        pred_scores = pred['scores'].numpy()
        for c in np.unique(full_mask)[np.unique(full_mask) > 0]:
            gt_binary = (full_mask == c).astype(np.uint8)
            idx_c = np.where(pred_labels == c)[0]
            if len(idx_c) == 0:
                pred_binary = np.zeros_like(gt_binary)
            else:
                best = idx_c[np.argmax(pred_scores[idx_c])]
                pred_binary = (pred_masks[best, 0] > 0.5).astype(np.uint8)
            scores[c].append(metric_fn(pred_binary, gt_binary))
    return {c: float(np.mean(v)) for c, v in scores.items() if v}


def compute_dice_per_class(predictions, targets, num_classes=NUM_CLASSES):
    """Dice por clase sobre test set. Excluye clases ausentes en cada imagen."""
    return _scores_per_class(predictions, targets, compute_dice, num_classes)


def compute_mean_dice(dice_per_class):
    return float(np.mean(list(dice_per_class.values()))) if dice_per_class else 0.0


def compute_miou(predictions, targets, num_classes=NUM_CLASSES):
    iou_per_class = _scores_per_class(predictions, targets, compute_iou, num_classes)
    return float(np.mean(list(iou_per_class.values()))) if iou_per_class else 0.0

### Métricas de detección (AP)

In [38]:
def _compute_ap(recalls, precisions):
    recalls    = np.concatenate(([0.0], recalls,    [1.0]))
    precisions = np.concatenate(([1.0], precisions, [0.0]))
    for i in range(len(precisions) - 2, -1, -1):
        precisions[i] = max(precisions[i], precisions[i + 1])
    idx = np.where(recalls[1:] != recalls[:-1])[0] + 1
    return float(np.sum((recalls[idx] - recalls[idx - 1]) * precisions[idx]))


def compute_ap50_per_class(predictions, targets, num_classes=NUM_CLASSES, iou_threshold=0.5):
    """AP@50 por clase. TP si IoU ≥ iou_threshold. Cada GT emparejado como máximo una vez."""
    ap_per_class = {}
    for c in range(1, num_classes):
        all_entries, n_gt = [], 0
        for pred, target in zip(predictions, targets):
            full_mask   = target['full_mask'].numpy()
            gt_binary   = (full_mask == c).astype(bool)
            has_gt      = gt_binary.any()
            if has_gt:
                n_gt += 1
            pred_labels = pred['labels'].numpy()
            pred_masks  = pred['masks'].numpy()
            pred_scores = pred['scores'].numpy()
            idx_c = np.where(pred_labels == c)[0]
            if len(idx_c) == 0:
                continue
            gt_matched = False
            for i in sorted(idx_c, key=lambda x: -pred_scores[x]):
                pm = (pred_masks[i, 0] > 0.5).astype(bool)
                if has_gt and not gt_matched:
                    inter = (pm & gt_binary).sum()
                    union = (pm | gt_binary).sum()
                    if union > 0 and inter / union >= iou_threshold:
                        all_entries.append((pred_scores[i], 1))
                        gt_matched = True
                        continue
                all_entries.append((pred_scores[i], 0))
        if n_gt == 0:
            continue
        if not all_entries:
            ap_per_class[c] = 0.0
            continue
        all_entries.sort(key=lambda x: -x[0])
        tp_arr  = np.array([e[1] for e in all_entries])
        tp_cum  = np.cumsum(tp_arr)
        fp_cum  = np.cumsum(1 - tp_arr)
        recalls = tp_cum / n_gt
        precs   = tp_cum / (tp_cum + fp_cum)
        ap_per_class[c] = _compute_ap(recalls, precs)
    return ap_per_class


def compute_map(ap_per_class):
    return float(np.mean(list(ap_per_class.values()))) if ap_per_class else 0.0

### Evaluación completa

In [39]:
def run_inference(model, dataloader, device):
    """Inferencia sobre el test set. Retorna (predictions, targets) en CPU."""
    model.eval()
    all_predictions, all_targets = [], []
    with torch.no_grad():
        for images, targets in dataloader:
            images  = [img.to(device) for img in images]
            outputs = model(images)
            for output, target in zip(outputs, targets):
                all_predictions.append({k: v.cpu() for k, v in output.items()})
                all_targets.append({k: v.cpu() for k, v in target.items()})
    return all_predictions, all_targets


def evaluate_model(predictions, targets, num_classes=NUM_CLASSES):
    """Calcula todas las métricas obligatorias sobre el test set."""
    dice_per_class = compute_dice_per_class(predictions, targets, num_classes)
    iou_per_class  = _scores_per_class(predictions, targets, compute_iou, num_classes)
    ap50_per_class = compute_ap50_per_class(predictions, targets, num_classes)
    return {
        'dice_per_class': dice_per_class,
        'iou_per_class':  iou_per_class,
        'mean_dice':      compute_mean_dice(dice_per_class),
        'miou':           compute_miou(predictions, targets, num_classes),
        'ap50_per_class': ap50_per_class,
        'map':            compute_map(ap50_per_class),
    }


def print_metrics_report(metrics):
    """Tabla resumen de métricas por clase y globales."""
    header = f"{'Clase':<8} {'Dice':>8} {'IoU':>8} {'AP@50':>8}"
    print(header)
    print('-' * len(header))
    for c in range(1, NUM_CLASSES):
        name = CLASS_NAMES.get(c, f'ID{c}')
        dice = metrics['dice_per_class'].get(c, float('nan'))
        iou  = metrics['iou_per_class'].get(c, float('nan'))
        ap50 = metrics['ap50_per_class'].get(c, float('nan'))
        print(f"{name:<8} {dice:>8.4f} {iou:>8.4f} {ap50:>8.4f}")
    print('-' * len(header))
    print(f"{'mean':<8} {metrics['mean_dice']:>8.4f} {metrics['miou']:>8.4f} {metrics['map']:>8.4f}")
    print()
    print(f"mean Dice: {metrics['mean_dice']:.4f}")
    print(f"mIoU:      {metrics['miou']:.4f}")
    print(f"mAP@50:    {metrics['map']:.4f}")

---
## Pipeline

In [ ]:
# === PREPROCESAMIENTO ===
index_df = load_dataset_index(DATASET_INDEX)
train_df, val_df, test_df = split_dataset(index_df, seed=SEED)
print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')

aug_pipeline = build_augmentation_pipeline()
train_ds     = SpineDataset(train_df, mode='train', aug_pipeline=aug_pipeline)
val_ds       = SpineDataset(val_df,   mode='val')
test_ds      = SpineDataset(test_df,  mode='test')
train_dl, val_dl, test_dl = create_dataloaders(train_ds, val_ds, test_ds)

# === PROCESAMIENTO (ENTRENAMIENTO) ===
model = build_model(NUM_CLASSES).to(DEVICE)
model = run_progressive_training(model, train_dl, val_dl, DEVICE)

# === GUARDAR MODELO FINAL ===
save_final_model(model, os.path.join(MODELS_DIR, 'maskrcnn_efficientnet_fpn_best.pth'))

# === MÉTRICAS ===
predictions, targets = run_inference(model, test_dl, DEVICE)
metrics = evaluate_model(predictions, targets)
print_metrics_report(metrics)

Train: 175  Val: 37  Test: 38


C:\Users\USER\AppData\Local\Temp\ipykernel_25292\2009201848.py:5: UserWarning: Argument(s) 'value, mask_value' are not valid for transform Rotate
  A.Rotate(limit=10, border_mode=cv2.BORDER_CONSTANT, value=0, mask_value=0, p=0.5),
Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.
C:\Users\USER\AppData\Local\Temp\ipykernel_25292\3430980908.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler, best_val_loss, epochs_no_impr = torch.cuda.amp.GradScaler(), float('inf'), 0
c:\Users\USER\segmentacion-vertebras-proyecto-maia\.venv\lib\site-packages\torch\amp\grad_scaler.py:136: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(



=== Fase 1 — Encoder congelado ===


In [ ]:
aug_pipeline = build_augmentation_pipeline()

train_ds = SpineDataset(train_df, mode="train", aug_pipeline=aug_pipeline)
val_ds   = SpineDataset(val_df, mode="val")
test_ds  = SpineDataset(test_df, mode="test")

print(len(train_ds), len(val_ds), len(test_ds))

In [ ]:
image, target = train_ds[2]

print("Imagen:", image.shape)
print("Target keys:", target.keys())
print("Boxes:", target["boxes"].shape)
print("Masks:", target["masks"].shape)
print("Labels:", target["labels"])
print("Full mask:", target["full_mask"].shape)

In [ ]:
print(index_df.columns)
print(index_df.head())